# Sequences Before Attention: RNNs, Vanishing Gradients, and Why Transformers Won

The runnable code of [Sequences Before Attention: RNNs, Vanishing Gradients, and Why Transformers Won](https://residual-stream.lvtan71.workers.dev/posts/rnns-and-why-transformers-won/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/rnns-and-why-transformers-won.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 100 min end to end.
Trains about thirty small recurrent models; the longest part of the series' Part III to rerun.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "rnns-and-why-transformers-won"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/rnns-and-why-transformers-won/charlm.py', 'shared/rnns-and-why-transformers-won/seq2seq.py', 'data/blog_train.txt', 'data/blog_val.txt']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## A state that carries the past

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/rnns-and-why-transformers-won/#a-state-that-carries-the-past)

In [ ]:
import numpy as np

# An Elman RNN with a 2-number state, run for 3 steps:
#   h_t = tanh(x_t W_x + h_{t-1} W_h + b),   and a prediction y = h_3 w_out at the end.
W_x = np.array([[0.5, -0.3]])                # one input number per step
W_h = np.array([[0.8, 0.2], [-0.4, 0.6]])    # the same matrix at every step
b, w_out = np.array([0.1, 0.0]), np.array([1.0, -1.0])
xs = [1.0, 0.0, -1.0]

def forward(W_h):
    h, hs = np.zeros(2), [np.zeros(2)]
    for x in xs:
        h = np.tanh(x * W_x[0] + h @ W_h + b)
        hs.append(h)
    return hs, hs[-1] @ w_out

hs, y = forward(W_h)
for t, h in enumerate(hs[1:], 1):
    print(f"h_{t} = {np.round(h, 4)}")
print(f"prediction y = h_3 . w_out = {y:.4f}")

# Backprop through time: run "Backpropagation, by Hand"'s rules backward through the 3 steps.
# The same W_h was used at every step, so its gradient is the sum over steps.
dW_h, dh = np.zeros_like(W_h), w_out.copy()          # dy/dh_3
for t in range(3, 0, -1):
    dz = dh * (1 - hs[t] ** 2)                        # back through tanh
    dW_h += np.outer(hs[t - 1], dz)                   # this step's share of the gradient
    dh = dz @ W_h.T                                   # pass back to h_{t-1}
    print(f"gradient reaching h_{t - 1}: {np.round(dh, 4)}")
eps, numeric = 1e-6, np.zeros_like(W_h)
for i in np.ndindex(W_h.shape):
    E = np.zeros_like(W_h); E[i] = eps
    numeric[i] = (forward(W_h + E)[1] - forward(W_h - E)[1]) / (2 * eps)
print(f"dy/dW_h = {np.round(dW_h, 4).tolist()}; largest gap to finite differences "
      f"{np.abs(dW_h - numeric).max():.1e}")

The post printed:

```text
h_1 = [ 0.537  -0.2913]
h_2 = [ 0.5691 -0.0673]
h_3 = [0.082 0.357]
prediction y = h_3 . w_out = -0.2750
gradient reaching h_2: [ 0.6201 -0.9208]
gradient reaching h_1: [ 0.1521 -0.7177]
gradient reaching h_0: [-0.0448 -0.4374]
dy/dW_h = [[0.7904, -0.9888], [-0.189, 0.3257]]; largest gap to finite differences 4.6e-11
```

## The vanishing gradient, measured

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/rnns-and-why-transformers-won/#the-vanishing-gradient-measured)

In [ ]:
import numpy as np

# How much of the gradient at step 50 survives back to step 0?
# Backprop multiplies by W_h^T (times tanh's slope) once per step, 50 times.
rng = np.random.default_rng(0)
d = 64
Q, _ = np.linalg.qr(rng.normal(size=(d, d)))          # a rotation: every direction the same size

def run(radius, act, input_size=0.5):
    W = radius * Q                                     # every singular value equals `radius`
    h, hs = rng.normal(size=d) * 0.5, []
    for _ in range(50):                                # forward, keeping the states
        z = h @ W + rng.normal(size=d) * input_size    # plus a fresh input each step
        h = np.tanh(z) if act == "tanh" else z
        hs.append(h)
    g, sizes = np.ones(d) / np.sqrt(d), {}
    for t in range(49, -1, -1):                        # backward from step 50
        if act == "tanh":
            g = g * (1 - hs[t] ** 2)
        g = g @ W.T
        if 50 - t in (1, 10, 25, 50):
            sizes[50 - t] = np.linalg.norm(g)
    return sizes

for radius in (0.9, 1.0, 1.1, 2.0):
    for act in ("linear", "tanh"):
        if act == "linear" and radius == 2.0:
            continue                                   # 2^50: no need to print it
        s = run(radius, act)
        print(f"radius {radius}, {act:6s}: gradient size after " +
              ", ".join(f"{k} steps {v:.1e}" for k, v in s.items()))
s = run(1.1, "tanh", input_size=0.0)
print("radius 1.1, tanh, no inputs: gradient size after " + ", ".join(f"{k} steps {v:.1e}" for k, v in s.items()))

The post printed:

```text
radius 0.9, linear: gradient size after 1 steps 9.0e-01, 10 steps 3.5e-01, 25 steps 7.2e-02, 50 steps 5.2e-03
radius 0.9, tanh  : gradient size after 1 steps 6.6e-01, 10 steps 1.6e-02, 25 steps 1.2e-04, 50 steps 1.6e-08
radius 1.0, linear: gradient size after 1 steps 1.0e+00, 10 steps 1.0e+00, 25 steps 1.0e+00, 50 steps 1.0e+00
radius 1.0, tanh  : gradient size after 1 steps 7.4e-01, 10 steps 6.1e-02, 25 steps 1.1e-03, 50 steps 1.8e-06
radius 1.1, linear: gradient size after 1 steps 1.1e+00, 10 steps 2.6e+00, 25 steps 1.1e+01, 50 steps 1.2e+02
radius 1.1, tanh  : gradient size after 1 steps 8.4e-01, 10 steps 1.8e-01, 25 steps 5.3e-03, 50 steps 2.8e-05
radius 2.0, tanh  : gradient size after 1 steps 1.1e+00, 10 steps 2.9e+00, 25 steps 1.3e+01, 50 steps 1.1e+02
radius 1.1, tanh, no inputs: gradient size after 1 steps 1.0e+00, 10 steps 9.8e-01, 25 steps 1.0e+00, 50 steps 8.7e-01
```

## Gates: LSTM and GRU

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/rnns-and-why-transformers-won/#gates-lstm-and-gru)

In [ ]:
import torch

# How much does the input at step 0 affect the state at step T? The size of
# d h_T / d x_0 for untrained cells of the same width, at initialization.
# The gates' biases decide how much a cell "starts out remembering": the
# LSTM's forget gate (PyTorch order: input, forget, cell, output) and the
# GRU's update gate (order: reset, update, new) keep the old state when near 1.
T, d = 50, 64

def gradient_back(cls, gate_bias=None):
    sizes = []
    for seed in range(5):
        torch.manual_seed(seed)
        cell = cls(d, d, batch_first=True)
        if gate_bias is not None:
            with torch.no_grad():
                cell.bias_ih_l0[d:2 * d].fill_(gate_bias); cell.bias_hh_l0[d:2 * d].fill_(0.0)
        x = torch.randn(1, T, d, requires_grad=True)
        out, _ = cell(x)
        out[0, -1].sum().backward()
        sizes.append([x.grad[0, t].norm().item() for t in (T - 1, T - 10, T - 25, 0)])
    return torch.tensor(sizes).median(0).values

for name, cls, bias in [("RNN", torch.nn.RNN, None),
                        ("GRU, default", torch.nn.GRU, None), ("GRU, update bias +1", torch.nn.GRU, 1.0),
                        ("LSTM, forget bias 0", torch.nn.LSTM, 0.0), ("LSTM, forget bias 1", torch.nn.LSTM, 1.0),
                        ("LSTM, forget bias 3", torch.nn.LSTM, 3.0)]:
    s = gradient_back(cls, bias)
    print(f"{name:20s}: gradient reaching the input 1 / 10 / 25 / 50 steps back: "
          + " / ".join(f"{v:.1e}" for v in s))

The post printed:

```text
RNN                 : gradient reaching the input 1 / 10 / 25 / 50 steps back: 3.5e+00 / 2.4e-03 / 1.4e-08 / 3.5e-17
GRU, default        : gradient reaching the input 1 / 10 / 25 / 50 steps back: 2.2e+00 / 1.6e-02 / 1.3e-05 / 6.2e-11
GRU, update bias +1 : gradient reaching the input 1 / 10 / 25 / 50 steps back: 1.3e+00 / 9.9e-02 / 1.7e-03 / 3.5e-06
LSTM, forget bias 0 : gradient reaching the input 1 / 10 / 25 / 50 steps back: 1.0e+00 / 6.7e-03 / 2.1e-06 / 8.1e-12
LSTM, forget bias 1 : gradient reaching the input 1 / 10 / 25 / 50 steps back: 1.0e+00 / 8.6e-02 / 3.1e-03 / 1.8e-05
LSTM, forget bias 3 : gradient reaching the input 1 / 10 / 25 / 50 steps back: 8.9e-01 / 4.1e-01 / 4.3e-01 / 4.6e-01
```

## A character model of this blog

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/rnns-and-why-transformers-won/#a-character-model-of-this-blog)

In [ ]:
import glob
import os
import re

# The blog's own English posts, as plain text: frontmatter, imports and
# component tags removed; prose, math and code kept. Two whole posts are held
# out for validation, so the model is scored on text it has never seen.
# Run once to freeze a snapshot (2026-09-30), so later posts don't change it.
POSTS = "content/posts"
HELD_OUT = {"kv-cache-explained", "bpe-tokenizer-explained"}      # validation posts

def clean(text):
    text = re.sub(r"\A---.*?---\n", "", text, flags=re.S)         # frontmatter
    text = re.sub(r"^import .*$", "", text, flags=re.M)           # MDX imports
    text = re.sub(r"<[A-Z][^>]*?/>", "", text, flags=re.S)        # self-closing components
    return re.sub(r"\n{3,}", "\n\n", text).strip() + "\n"

def load():
    train, val = [], []
    for path in sorted(glob.glob(f"{POSTS}/*.mdx")):
        slug = os.path.basename(path)[:-4]
        (val if slug in HELD_OUT else train).append(clean(open(path).read()))
    return "".join(train), "".join(val)

if __name__ == "__main__":
    if os.path.exists("blog_train.txt"):                      # use the frozen snapshot if it's there
        train, val = open("blog_train.txt").read(), open("blog_val.txt").read()
    else:
        train, val = load()
        open("blog_train.txt", "w").write(train)
        open("blog_val.txt", "w").write(val)
    chars = sorted(set(train))
    print(f"train {len(train):,} characters, validation {len(val):,}; {len(chars)} distinct characters")
    print(f"validation characters never seen in training: {sum(c not in set(train) for c in val)} "
          f"of {len(val):,}")

The post printed:

```text
train 593,616 characters, validation 56,786; 147 distinct characters
validation characters never seen in training: 123 of 56,786
```

In [ ]:
import math
from collections import Counter

# Baselines with no neural network at all: predict the next character from
# the previous k characters by counting, with add-one smoothing.
train, val = open("blog_train.txt").read(), open("blog_val.txt").read()
V = len(set(train)) + 1                                   # + an "unseen" character
for k in (0, 1, 2, 3, 4):
    ctx, pair = Counter(), Counter()
    for i in range(k, len(train)):
        ctx[train[i - k:i]] += 1
        pair[train[i - k:i + 1]] += 1
    nll = -sum(math.log((pair[val[i - k:i + 1]] + 1) / (ctx[val[i - k:i]] + V))
               for i in range(k, len(val))) / (len(val) - k)
    print(f"previous {k} characters: validation {nll:.3f} nats/char = {nll / math.log(2):.3f} bits/char")

The post printed:

```text
previous 0 characters: validation 3.468 nats/char = 5.003 bits/char
previous 1 characters: validation 2.849 nats/char = 4.110 bits/char
previous 2 characters: validation 2.481 nats/char = 3.580 bits/char
previous 3 characters: validation 2.628 nats/char = 3.791 bits/char
previous 4 characters: validation 3.112 nats/char = 4.490 bits/char
```

In [ ]:
import math
from collections import Counter, defaultdict

# Interpolated Kneser-Ney smoothing (Chen and Goodman, 1998), with a fixed
# discount of 0.75. Instead of spreading unseen probability evenly, it backs
# off to a shorter context, and the shortest level counts how many different
# contexts a character follows, not how often it occurs.
train, val = open("blog_train.txt").read(), open("blog_val.txt").read()
chars = sorted(set(train))
V, D = len(chars) + 1, 0.75                                # + one "unseen" character

def build(order):
    count = [Counter() for _ in range(order + 1)]          # count[n][context + char], n = context length
    for n in range(order + 1):
        for i in range(n, len(train)):
            count[n][train[i - n:i + 1]] += 1
    # For lower orders, KN uses continuation counts: in how many distinct contexts does it appear?
    cont = [Counter() for _ in range(order + 1)]
    for n in range(order):
        for gram in count[n + 1]:
            cont[n][gram[1:]] += 1
    tables = []
    for n in range(order + 1):
        c = count[n] if n == order else cont[n]
        total, types = defaultdict(int), defaultdict(int)
        for gram, k in c.items():
            total[gram[:-1]] += k
            types[gram[:-1]] += 1
        tables.append((c, total, types))
    return tables

def prob(tables, ctx, ch):
    p = 1 / V                                              # the uniform floor
    for n in range(len(tables)):
        c, total, types = tables[n]
        h = ctx[len(ctx) - n:] if n else ""
        if total[h] == 0:
            continue                                        # context never seen: keep the shorter estimate
        p = max(c[h + ch] - D, 0) / total[h] + D * types[h] / total[h] * p
    return p

for order in range(1, 8):
    tables = build(order)
    nll = -sum(math.log(prob(tables, val[max(0, i - order):i], val[i])) for i in range(1, len(val))) / (len(val) - 1)
    print(f"previous {order} characters: {nll / math.log(2):.3f} bits/char")
check = build(3)
print(f"probabilities over all characters sum to "
      f"{sum(prob(check, 'the', c) for c in chars) + prob(check, 'the', '\\x00'):.6f} after 'the'")

The post printed:

```text
previous 1 characters: 4.124 bits/char
previous 2 characters: 3.370 bits/char
previous 3 characters: 2.844 bits/char
previous 4 characters: 2.650 bits/char
previous 5 characters: 2.600 bits/char
previous 6 characters: 2.594 bits/char
previous 7 characters: 2.606 bits/char
probabilities over all characters sum to 1.000000 after 'the'
```

In [ ]:
%%writefile charlm.py
import math
import sys
import time
import torch

# A character-level language model of this blog: read 128 characters, predict
# each next one. Three recurrent cells, the same size, the same training.
train_text, val_text = open("blog_train.txt").read(), open("blog_val.txt").read()
chars = sorted(set(train_text))
index = {c: i + 1 for i, c in enumerate(chars)}           # 0 = a character never seen in training
encode = lambda s: torch.tensor([index.get(c, 0) for c in s])
train, val = encode(train_text), encode(val_text)
V = len(chars) + 1

class CharRNN(torch.nn.Module):
    def __init__(self, cell, hidden=256):
        super().__init__()
        self.embed = torch.nn.Embedding(V, 64)
        self.rnn = {"RNN": torch.nn.RNN, "GRU": torch.nn.GRU, "LSTM": torch.nn.LSTM}[cell](
            64, hidden, batch_first=True)
        self.out = torch.nn.Linear(hidden, V)
    def forward(self, x, state=None):
        h, state = self.rnn(self.embed(x), state)
        return self.out(h), state

def batch(data, rng, B=64, T=128):
    i = torch.randint(0, len(data) - T - 1, (B,), generator=rng)
    x = torch.stack([data[j:j + T] for j in i])
    return x, torch.stack([data[j + 1:j + T + 1] for j in i])

@torch.no_grad()
def evaluate(model, data, T=128):
    """Average loss per character over the whole text, carrying the state along."""
    total, state = 0.0, None
    for s in range(0, len(data) - 1 - T, T):
        logits, state = model(data[s:s + T][None], state)
        total += torch.nn.functional.cross_entropy(logits[0], data[s + 1:s + T + 1], reduction="sum").item()
    return total / ((len(data) - 1 - T) // T * T)

cell, seed, steps = sys.argv[1], int(sys.argv[2]), int(sys.argv[3]) if len(sys.argv) > 3 else 3000
torch.manual_seed(seed)
model, rng = CharRNN(cell), torch.Generator().manual_seed(seed)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0.0)
t0 = time.perf_counter()
for step in range(1, steps + 1):
    x, y = batch(train, rng)
    logits, _ = model(x)
    loss = torch.nn.functional.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)     # "Initialization and Optimizers"
    opt.step()
val_loss = evaluate(model, val)
train_loss = evaluate(model, train[:len(val)])                 # the same amount of training text
params = sum(p.numel() for p in model.parameters())
print(f"{cell:4s} seed {seed}, {steps:,} steps: {params:,} parameters, {time.perf_counter() - t0:.0f} s; "
      f"train {train_loss / math.log(2):.3f}, validation {val_loss / math.log(2):.3f} bits/char")

In [ ]:
# The post trained each cell type with three seeds:
for cell in ("RNN", "GRU", "LSTM"):
    for seed in (0, 1, 2):
        %run charlm.py {cell} {seed}

The post printed:

```text
RNN  seed 0, 3,000 steps: 129,940 parameters, 58 s; train 1.941, validation 2.839 bits/char
RNN  seed 1, 3,000 steps: 129,940 parameters, 52 s; train 1.946, validation 2.829 bits/char
RNN  seed 2, 3,000 steps: 129,940 parameters, 52 s; train 1.947, validation 2.832 bits/char
GRU  seed 0, 3,000 steps: 294,804 parameters, 141 s; train 1.547, validation 2.827 bits/char
GRU  seed 1, 3,000 steps: 294,804 parameters, 173 s; train 1.575, validation 2.842 bits/char
GRU  seed 2, 3,000 steps: 294,804 parameters, 168 s; train 1.615, validation 2.812 bits/char
LSTM seed 0, 3,000 steps: 377,236 parameters, 263 s; train 1.714, validation 2.694 bits/char
LSTM seed 1, 3,000 steps: 377,236 parameters, 268 s; train 1.729, validation 2.696 bits/char
LSTM seed 2, 3,000 steps: 377,236 parameters, 257 s; train 1.781, validation 2.690 bits/char
```

In [ ]:
for cell in ("RNN", "GRU", "LSTM"):
    %run charlm.py {cell} 0 12000

The post printed:

```text
RNN  seed 0, 12,000 steps: 129,940 parameters, 326 s; train 1.793, validation 3.017 bits/char
GRU  seed 0, 12,000 steps: 294,804 parameters, 737 s; train 1.351, validation 3.182 bits/char
LSTM seed 0, 12,000 steps: 377,236 parameters, 761 s; train 1.211, validation 3.369 bits/char
```

## The bottleneck, and attention

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/rnns-and-why-transformers-won/#the-bottleneck-and-attention)

In [ ]:
%%writefile seq2seq.py
import sys
import torch

# Reverse a sequence of random digits: an encoder reads it, a decoder writes it
# backwards. Without attention, the decoder sees only the encoder's last state;
# with attention, it can look back at every encoder state (Bahdanau et al.).
BOS, V, H = 10, 11, 128

class Seq2Seq(torch.nn.Module):
    def __init__(self, attention):
        super().__init__()
        self.attention = attention
        self.embed = torch.nn.Embedding(V, 32)
        self.encoder = torch.nn.GRU(32, H, batch_first=True)
        self.decoder = torch.nn.GRUCell(32 + (H if attention else 0), H)
        self.W_dec, self.W_enc = torch.nn.Linear(H, H), torch.nn.Linear(H, H)
        self.v = torch.nn.Linear(H, 1, bias=False)
        self.out = torch.nn.Linear(H, 10)

    def forward(self, src, tgt_in):
        enc, h = self.encoder(self.embed(src))                 # enc: every step; h: the last one
        h, logits = h[0], []
        for t in range(tgt_in.shape[1]):
            x = self.embed(tgt_in[:, t])
            if self.attention:                                 # additive attention over enc
                score = self.v(torch.tanh(self.W_dec(h)[:, None] + self.W_enc(enc)))[..., 0]
                context = (torch.softmax(score, dim=1)[..., None] * enc).sum(1)
                x = torch.cat([x, context], dim=1)
            h = self.decoder(x, h)
            logits.append(self.out(h))
        return torch.stack(logits, 1)

def data(n, L, g):
    src = torch.randint(0, 10, (n, L), generator=g)
    tgt = src.flip(1)
    return src, torch.cat([torch.full((n, 1), BOS), tgt[:, :-1]], 1), tgt

@torch.no_grad()
def greedy(model, src, L):
    y, out = torch.full((len(src), 1), BOS), []
    for _ in range(L):                                        # feed back its own predictions
        nxt = model(src, y)[:, -1].argmax(-1, keepdim=True)
        y = torch.cat([y, nxt], 1)
    return y[:, 1:]

L, attention, seed = int(sys.argv[1]), sys.argv[2] == "attention", int(sys.argv[3])
steps = int(sys.argv[4]) if len(sys.argv) > 4 else 2000
lr = float(sys.argv[5]) if len(sys.argv) > 5 else 3e-3
torch.manual_seed(seed)
g = torch.Generator().manual_seed(seed)
model = Seq2Seq(attention)
opt = torch.optim.Adam(model.parameters(), lr=lr)
for step in range(steps):
    src, tgt_in, tgt = data(64, L, g)
    loss = torch.nn.functional.cross_entropy(model(src, tgt_in).reshape(-1, 10), tgt.reshape(-1))
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
src, _, tgt = data(1000, L, torch.Generator().manual_seed(1000 + seed))   # fresh test sequences
pred = greedy(model, src, L)
print(f"length {L:2d}, {'attention' if attention else 'no attention':12s}, seed {seed}"
      f"{'' if steps == 2000 else f', {steps:,} steps'}{'' if lr == 3e-3 else f', learning rate {lr:g}'}: "
      f"digits right {(pred == tgt).float().mean():.3f}, whole sequences right {(pred == tgt).all(1).float().mean():.3f}")

In [ ]:
# Each length, with and without attention, two seeds, at the default 2,000 steps:
for L in (5, 10, 20, 40):
    for attn in ("none", "attention"):
        for seed in (0, 1):
            %run seq2seq.py {L} {attn} {seed}

The post printed:

```text
length  5, no attention, seed 0: digits right 0.990, whole sequences right 0.972
length  5, no attention, seed 1: digits right 0.994, whole sequences right 0.979
length  5, attention   , seed 0: digits right 0.998, whole sequences right 0.996
length  5, attention   , seed 1: digits right 1.000, whole sequences right 1.000
length 10, no attention, seed 0: digits right 0.962, whole sequences right 0.832
length 10, no attention, seed 1: digits right 0.971, whole sequences right 0.853
length 10, attention   , seed 0: digits right 0.987, whole sequences right 0.958
length 10, attention   , seed 1: digits right 0.984, whole sequences right 0.958
length 20, no attention, seed 0: digits right 0.784, whole sequences right 0.148
length 20, no attention, seed 1: digits right 0.818, whole sequences right 0.189
length 20, attention   , seed 0: digits right 0.938, whole sequences right 0.840
length 20, attention   , seed 1: digits right 0.944, whole sequences right 0.869
length 40, no attention, seed 0: digits right 0.374, whole sequences right 0.000
length 40, no attention, seed 1: digits right 0.379, whole sequences right 0.000
length 40, attention   , seed 0: digits right 0.479, whole sequences right 0.106
length 40, attention   , seed 1: digits right 0.250, whole sequences right 0.000
```

In [ ]:
for attn in ("none", "attention"):
    for seed in (0, 1):
        %run seq2seq.py 40 {attn} {seed} 3000

The post printed:

```text
length 40, no attention, seed 0, 3,000 steps: digits right 0.490, whole sequences right 0.000
length 40, no attention, seed 1, 3,000 steps: digits right 0.513, whole sequences right 0.000
length 40, attention   , seed 0, 3,000 steps: digits right 0.195, whole sequences right 0.000
length 40, attention   , seed 1, 3,000 steps: digits right 0.177, whole sequences right 0.000
```

In [ ]:
for attn in ("none", "attention"):
    for seed in (0, 1):
        %run seq2seq.py 40 {attn} {seed} 3000 1e-3

The post printed:

```text
length 40, no attention, seed 0, 3,000 steps, learning rate 0.001: digits right 0.224, whole sequences right 0.000
length 40, no attention, seed 1, 3,000 steps, learning rate 0.001: digits right 0.238, whole sequences right 0.000
length 40, attention   , seed 0, 3,000 steps, learning rate 0.001: digits right 0.971, whole sequences right 0.927
length 40, attention   , seed 1, 3,000 steps, learning rate 0.001: digits right 0.975, whole sequences right 0.940
```

In [ ]:
for seed in (0, 1):
    %run seq2seq.py 40 none {seed} 6000

The post printed:

```text
length 40, no attention, seed 0, 6,000 steps: digits right 0.701, whole sequences right 0.006
length 40, no attention, seed 1, 6,000 steps: digits right 0.745, whole sequences right 0.015
```

## Why Transformers won

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/rnns-and-why-transformers-won/#why-transformers-won)

In [ ]:
import time
import torch

# The same 1,024-token sequence at width 256: a GRU must take 1,024 steps one
# after another; self-attention does it as a few big matrix multiplies.
torch.manual_seed(0)
T, d = 1024, 256
x = torch.randn(1, T, d)
gru = torch.nn.GRU(d, d, batch_first=True)
Wq, Wk, Wv = (torch.randn(d, d) / d ** 0.5 for _ in range(3))

def attention(x):
    q, k, v = x @ Wq, x @ Wk, x @ Wv
    mask = torch.ones(T, T).triu(1).bool()                 # each token sees only the past
    s = (q @ k.transpose(-1, -2) / d ** 0.5).masked_fill(mask, float("-inf"))
    return torch.softmax(s, -1) @ v

def timed(fn, trials=10):
    fn(); times = []
    for _ in range(trials):
        t0 = time.perf_counter(); fn(); times.append(time.perf_counter() - t0)
    return sorted(times)[trials // 2]

with torch.no_grad():
    t_rnn, t_att = timed(lambda: gru(x)), timed(lambda: attention(x))
flops_rnn = T * 3 * 2 * 2 * d * d                           # 3 gates, input and hidden matmuls
flops_att = 3 * 2 * T * d * d + 2 * 2 * T * T * d           # projections, then scores and weighted sum
print(f"GRU:       {T} sequential steps, {flops_rnn / 1e9:.2f} GFLOPs, {t_rnn * 1e3:6.1f} ms")
print(f"attention: 1 parallel pass,      {flops_att / 1e9:.2f} GFLOPs, {t_att * 1e3:6.1f} ms")

The post printed:

```text
GRU:       1024 sequential steps, 0.81 GFLOPs,    9.5 ms
attention: 1 parallel pass,      1.48 GFLOPs,    1.5 ms
```